### RAG

In [2]:
import os
from operator import itemgetter

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

load_dotenv()


True

In [9]:
print("Initializing components...")

embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-5.2")

vectorstore = PineconeVectorStore(
    index_name=os.environ["INDEX_NAME"], embedding=embeddings
)


Initializing components...


### Vector Store

In [ ]:
K=5

vs_retriever = vectorstore.as_retriever(search_kwargs={"k": K})

In [11]:
prompt_template = ChatPromptTemplate.from_template(
"""Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer:"""
)


In [ ]:

def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)


# ============================================================================
# IMPLEMENTATION 1: Without LCEL (Simple Function-Based Approach)
# ============================================================================
def retrieval_chain_without_lcel(query: str):
    """
    Simple retrieval chain without LCEL.
    Manually retrieves documents, formats them, and generates a response.

    Limitations:
    - Manual step-by-step execution
    - No built-in streaming support
    - No async support without additional code
    - Harder to compose with other chains
    - More verbose and error-prone
    """
    # Step 1: Retrieve relevant documents
    docs = vs_retriever.invoke(query)

    # Step 2: Format documents into context string
    context = format_docs(docs)

    # Step 3: Format the prompt with context and question
    messages = prompt_template.format_messages(context=context, question=query)

    # Step 4: Invoke LLM with the formatted messages
    response = llm.invoke(messages)

    # Step 5: Return the content
    return response.content


# ============================================================================
# IMPLEMENTATION 2: With LCEL (LangChain Expression Language) - BETTER APPROACH
# ============================================================================
def create_retrieval_chain_with_lcel():
    """
    Create a retrieval chain using LCEL (LangChain Expression Language).
    Returns a chain that can be invoked with {"question": "..."}

    Advantages over non-LCEL approach:
    - Declarative and composable: Easy to chain operations with pipe operator (|)
    - Built-in streaming: chain.stream() works out of the box
    - Built-in async: chain.ainvoke() and chain.astream() available
    - Batch processing: chain.batch() for multiple inputs
    - Type safety: Better integration with LangChain's type system
    - Less code: More concise and readable
    - Reusable: Chain can be saved, shared, and composed with other chains
    - Better debugging: LangChain provides better observability tools
    """
    retrieval_chain = (
        RunnablePassthrough.assign(
            context=itemgetter("question") | vs_retriever | format_docs
        )
        | prompt_template
        | llm
        | StrOutputParser()
    )
    return retrieval_chain


### Option 0: Raw invocation without RAG

In [13]:
print("Retrieving...")

# Query
query = "what is Pinecone in machine learning?"

print("\n" + "=" * 70)
print("IMPLEMENTATION 0: Raw LLM Invocation (No RAG)")
print("=" * 70)
result_raw = llm.invoke([HumanMessage(content=query)])
print("\nAnswer:")
print(result_raw.content)

Retrieving...

IMPLEMENTATION 0: Raw LLM Invocation (No RAG)

Answer:
Pinecone is a **managed vector database** (often used in machine learning for **semantic search and Retrieval-Augmented Generation, RAG**).

### What it does
- Stores and indexes **vector embeddings** (numeric representations of text, images, audio, etc.).
- Lets you do fast **nearest-neighbor search**: given a query embedding, it finds the most similar vectors.
- Supports **metadata filtering** (e.g., “only search documents from 2024”).
- Handles scaling, uptime, and performance so you don’t run the infrastructure yourself.

### Why it’s used in ML
Common use cases:
- **Semantic search** (search by meaning, not keywords)
- **RAG for LLM apps** (retrieve relevant documents and feed them to a language model)
- **Recommendation systems**
- **Deduplication / similarity detection**
- **Clustering / anomaly detection** (as a building block)

### Typical workflow
1. Create embeddings for your data (e.g., using OpenAI, Sent

### Option 1: Use implementation WITHOUT LCEL

In [14]:
print("\n" + "=" * 70)
print("IMPLEMENTATION 1: Without LCEL")
print("=" * 70)
result_without_lcel = retrieval_chain_without_lcel(query)
print("\nAnswer:")
print(result_without_lcel)



IMPLEMENTATION 1: Without LCEL

Answer:
Pinecone, in the context of machine learning, is a **fully managed, cloud-based vector database** used to **store and retrieve vector embeddings** so ML systems can quickly find **similar data points** (nearest neighbors) based on their vector representations.

From the provided context, Pinecone is designed for building and deploying **large-scale ML applications**, with key characteristics:

- **Fast and scalable similarity search:** Efficiently retrieves similar items using their vectors, even at very large scale (it can handle **millions or billions** of data points).
- **High throughput and low latency:** Built to support **high query volume** while keeping searches fast.
- **Fully managed infrastructure:** Pinecone provides **infrastructure management and maintenance**, so users don’t have to operate the underlying database systems themselves.
- **Real-time updates:** Supports **real-time syncing and updates** as new data points are added,

### Option 2: Use implementation WITH LCEL (Better Approach)

In [16]:
print("\n" + "=" * 70)
print("IMPLEMENTATION 2: With LCEL - Better Approach")
print("=" * 70)
print("Why LCEL is better:")
print("- More concise and declarative")
print("- Built-in streaming: chain.stream()")
print("- Built-in async: chain.ainvoke()")
print("- Easy to compose with other chains")
print("- Better for production use")
print("=" * 70)

chain_with_lcel = create_retrieval_chain_with_lcel()
result_with_lcel = chain_with_lcel.invoke({"question": query})
print("\nAnswer:")
print(result_with_lcel)




IMPLEMENTATION 2: With LCEL - Better Approach
Why LCEL is better:
- More concise and declarative
- Built-in streaming: chain.stream()
- Built-in async: chain.ainvoke()
- Easy to compose with other chains
- Better for production use

Answer:
Pinecone (in machine learning) is a **fully managed, cloud-based vector database** used to **store and retrieve vector embeddings** so ML systems can quickly find **similar data points** by comparing their vector representations.

Based on the provided context, Pinecone is designed for building and deploying **large-scale ML applications** and is characterized by:

- **Fast, scalable similarity search:** Efficient retrieval of similar items using vectors, and it can handle **millions to billions** of data points.
- **High throughput and low latency:** Built to support **high query volume** while maintaining **low-latency** search performance.
- **Fully managed infrastructure:** Pinecone handles **infrastructure management and maintenance**, reducin